# Exploration de l'historique des réservations

Tâches 1 à 3 : connexion à la base, exploration SQL, chargement et découpage des données.

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))  # pour pouvoir importer model.train_baseline plus tard
load_dotenv(ROOT / ".env")

engine = create_engine(os.environ["DATABASE_URL"])

## 1. Connexion à la base

In [2]:
pd.read_sql(text("SELECT COUNT(*) AS nb_reservations FROM reservations"), engine)

,nb_reservations
0,78703


## 2. Exploration SQL

Les agrégations sont calculées par PostgreSQL. `AVG(is_canceled)` donne le taux d'annulation.

In [3]:
pd.read_sql(text("SELECT * FROM hotels"), engine)

,hotel_id,code,nom,type,ville
0,1,City Hotel,Escale Atlantique Centre,urbain,Lisbonne
1,2,Resort Hotel,Escale Atlantique Océan,resort,Algarve


In [4]:
pd.read_sql(text("SELECT * FROM reservations LIMIT 5"), engine)

,reservation_id,hotel_id,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,...,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,arrival_date
0,1,1,0,6,2015,July,27,1,0,2,...,6,None,0,Transient,0.0,0,0,Check-Out,2015-07-03,2015-07-01
1,2,1,1,88,2015,July,27,1,0,4,...,9,None,0,Transient,76.5,0,1,Canceled,2015-07-01,2015-07-01
2,3,1,1,65,2015,July,27,1,0,4,...,9,None,0,Transient,68.0,0,1,Canceled,2015-04-30,2015-07-01
3,4,1,1,92,2015,July,27,1,2,4,...,9,None,0,Transient,76.5,0,2,Canceled,2015-06-23,2015-07-01
4,5,1,1,257,2015,July,27,1,0,2,...,6,None,0,Transient,101.5,0,0,Canceled,2015-06-16,2015-07-01


### Par hôtel

In [5]:
query = text("""
    SELECT h.nom,
           h.code,
           COUNT(*)                              AS nb_reservations,
           ROUND(AVG(r.is_canceled)::numeric, 3) AS taux_annulation
    FROM reservations AS r
    JOIN hotels AS h ON h.hotel_id = r.hotel_id
    GROUP BY h.nom, h.code
    ORDER BY nb_reservations DESC
""")
pd.read_sql(query, engine)

,nom,code,nb_reservations,taux_annulation
0,Escale Atlantique Centre,City Hotel,51822,0.413
1,Escale Atlantique Océan,Resort Hotel,26881,0.263


### Par mois d'arrivée

In [6]:
query = text("""
    SELECT EXTRACT(MONTH FROM arrival_date)::int AS num_mois,
           arrival_date_month                    AS mois,
           COUNT(*)                              AS nb_reservations,
           ROUND(AVG(is_canceled)::numeric, 3)   AS taux_annulation
    FROM reservations
    GROUP BY num_mois, mois
    ORDER BY num_mois
""")
pd.read_sql(query, engine)

,num_mois,mois,nb_reservations,taux_annulation
0,1,January,2248,0.248
1,2,February,3891,0.344
2,3,March,4824,0.306
3,4,April,5428,0.380
4,5,May,5478,0.350
5,6,June,5292,0.396
6,7,July,7348,0.375
7,8,August,8952,0.382
8,9,September,10508,0.392
9,10,October,11160,0.380


### Par délai de réservation

In [7]:
query = text("""
    SELECT CASE
               WHEN lead_time <= 7   THEN '1. 0-7 jours'
               WHEN lead_time <= 30  THEN '2. 8-30 jours'
               WHEN lead_time <= 90  THEN '3. 31-90 jours'
               WHEN lead_time <= 180 THEN '4. 91-180 jours'
               WHEN lead_time <= 365 THEN '5. 181-365 jours'
               ELSE '6. plus d''un an'
           END                                 AS delai,
           COUNT(*)                            AS nb_reservations,
           ROUND(AVG(is_canceled)::numeric, 3) AS taux_annulation
    FROM reservations
    GROUP BY delai
    ORDER BY delai
""")
pd.read_sql(query, engine)

,delai,nb_reservations,taux_annulation
0,1. 0-7 jours,13925,0.100
1,2. 8-30 jours,12602,0.265
2,3. 31-90 jours,20140,0.355
3,4. 91-180 jours,15748,0.434
4,5. 181-365 jours,14504,0.577
5,6. plus d'un an,1784,0.776


### Par segment et par type d'acompte

In [8]:
query = text("""
    SELECT market_segment,
           COUNT(*)                            AS nb_reservations,
           ROUND(AVG(is_canceled)::numeric, 3) AS taux_annulation
    FROM reservations
    GROUP BY market_segment
    ORDER BY nb_reservations DESC
""")
pd.read_sql(query, engine)

,market_segment,nb_reservations,taux_annulation
0,Online TA,33826,0.339
1,Offline TA/TO,18552,0.345
2,Groups,13957,0.623
3,Direct,7977,0.153
4,Corporate,3733,0.159
5,Complementary,529,0.117
6,Aviation,127,0.228
7,Undefined,2,1.000


In [9]:
query = text("""
    SELECT deposit_type,
           COUNT(*)                            AS nb_reservations,
           ROUND(AVG(is_canceled)::numeric, 3) AS taux_annulation
    FROM reservations
    GROUP BY deposit_type
    ORDER BY nb_reservations DESC
""")
pd.read_sql(query, engine)

,deposit_type,nb_reservations,taux_annulation
0,No Deposit,68107,0.266
1,Non Refund,10461,0.992
2,Refundable,135,0.074


### Observations

- Le City Hotel annule nettement plus que le Resort Hotel (41 % contre 26 %).
- Le taux d'annulation augmente avec le délai de réservation : 10 % sous 7 jours, 78 % au-delà d'un an.
- Les réservations `Non Refund` sont annulées à 99 %, alors qu'un acompte non remboursable devrait dissuader d'annuler.
- Le segment Groups annule à 62 %, les segments Direct et Corporate autour de 15 %.
- La saisonnalité est modérée : 25 % en janvier, autour de 38 % en été et en automne.

## 3. Chargement et découpage

La requête et le découpage sont définis dans `train.py` et réutilisés ici. La jointure sur `hotels` fournit la colonne `hotel` attendue par la recette v1.

In [10]:
from train import QUERY, load_data, split_train_test, TEST_MONTHS

print(QUERY)
df = load_data(engine)
print(df.shape)
df[["hotel", "hotel_id", "arrival_date", "is_canceled"]].head()


SELECT h.code AS hotel,
       r.*
FROM reservations AS r
JOIN hotels AS h ON h.hotel_id = r.hotel_id
ORDER BY r.arrival_date, r.reservation_id



(78703, 35)


,hotel,hotel_id,arrival_date,is_canceled
0,City Hotel,1,2015-07-01,0
1,City Hotel,1,2015-07-01,1
2,City Hotel,1,2015-07-01,1
3,City Hotel,1,2015-07-01,1
4,City Hotel,1,2015-07-01,1


### Découpage temporel

Le modèle s'applique à des réservations futures. Les 4 derniers mois d'arrivée constituent le jeu de test, qui n'est utilisé qu'à la tâche 8.

In [11]:
train, test = split_train_test(df)

resume = pd.DataFrame({
    "periode_arrivee": [f"{train.arrival_date.min():%Y-%m-%d} → {train.arrival_date.max():%Y-%m-%d}",
                        f"{test.arrival_date.min():%Y-%m-%d} → {test.arrival_date.max():%Y-%m-%d}"],
    "nb_reservations": [len(train), len(test)],
    "taux_annulation": [round(train.is_canceled.mean(), 3), round(test.is_canceled.mean(), 3)],
}, index=["entraînement", "test"])
resume

,periode_arrivee,nb_reservations,taux_annulation
entraînement,2015-07-01 → 2016-08-31,58792,0.356
test,2016-09-01 → 2016-12-31,19911,0.380
